In [ ]:
source("helpers/init.r")

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
validate_columns(cohorts, c("sampleId", "cohort", "ref", "primaryTumorLocation"), "cohorts")
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 1 - Schema

In [ ]:
location    <- cohorts %>% filter(!ref) %>% pull(primaryTumorLocation) %>% unique()
plt_title   <- cohorts %>% filter(!ref) %>% head(1) %>% pull(title)
schema_file <- if (!is.null(plt_title) && plt_title %in% names(title_schema_map))
  title_schema_map[[plt_title]] else schema_map[[location]]
message("Schema: ", schema_file, " (title override: ", plt_title %in% names(title_schema_map), ")")
cohort_n  <- cohorts %>% count(cohort, name = "ct")
lbl_pos   <- get_schema_label_pos(schema_file)
schema <-
  ggit(paste0(REFS_DIR, "schemas/", schema_file, ".png")) +
  annotate("text", x = lbl_pos$x_left,  y = lbl_pos$y_left,  size = 6, color = fills[["plt"]],
           label = paste0(plt_cohort, "\n", cohort_n %>% filter(cohort == plt_cohort) %>% pull(ct), " Patients")) +
  annotate("text", x = lbl_pos$x_right, y = lbl_pos$y_right, size = 6, color = fills[["ref"]],
           label = paste0(ref_cohort,  "\n", cohort_n %>% filter(cohort == ref_cohort)  %>% pull(ct), " Patients")) +
  coord_cartesian(xlim = c(0,1), ylim = c(0, 2.5), clip = "off")

# 2 - Save

In [ ]:
figures <- list(schema = schema)
saveRDS(figures, paste0(READY_FIG, "schemas.rds"), compress = FALSE)